In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("/Workspace/ETL project 1/finance_transactions.csv") \
    .getOrCreate()

In [0]:
df = spark.read.csv(
    "/Workspace/ETL project 1/finance_transactions.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show(10)

+--------------+----------------+-------------------+--------------------+--------------------+--------+--------+--------------+-----------+---------+--------------+
|transaction_id|transaction_date|         department|    transaction_type|     vendor_customer|  amount|currency|payment_method|cost_center|   status|invoice_number|
+--------------+----------------+-------------------+--------------------+--------------------+--------+--------+--------------+-----------+---------+--------------+
|       TXN0001|      2025-02-27|        Procurement|     Invoice Payment| BluePeak Consulting|15096.33|     INR| Bank Transfer|      CC500|Completed|     INV-21395|
|       TXN0002|      2025-08-05|   Accounts Payable|     Invoice Payment|    ABC Technologies| 8799.72|     INR|   Credit Card|      CC200|Completed|     INV-95181|
|       TXN0003|      2025-10-07|Accounts Receivable|       Bank Transfer|    ABC Technologies|  205.55|     USD|           UPI|      CC400|Completed|     INV-54597|
|   

In [0]:
#Check total rows and columns
print("Total rows:", df.count())
print("Total columns:", len(df.columns))
print("Column names:", df.columns)

Total rows: 500
Total columns: 11
Column names: ['transaction_id', 'transaction_date', 'department', 'transaction_type', 'vendor_customer', 'amount', 'currency', 'payment_method', 'cost_center', 'status', 'invoice_number']


In [0]:
#Find missing values
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+--------------+----------------+----------+----------------+---------------+------+--------+--------------+-----------+------+--------------+
|transaction_id|transaction_date|department|transaction_type|vendor_customer|amount|currency|payment_method|cost_center|status|invoice_number|
+--------------+----------------+----------+----------------+---------------+------+--------+--------------+-----------+------+--------------+
|             0|               0|         0|               0|             11|     7|       0|             0|          8|     0|             0|
+--------------+----------------+----------+----------------+---------------+------+--------+--------------+-----------+------+--------------+



In [0]:
#Check duplicate records
print("Total rows:", df.count())
print("Distinct rows:", df.distinct().count())

Total rows: 500
Distinct rows: 495


In [0]:
#Inspect sample records
display(df.limit(10))

transaction_id,transaction_date,department,transaction_type,vendor_customer,amount,currency,payment_method,cost_center,status,invoice_number
TXN0001,2025-02-27,Procurement,Invoice Payment,BluePeak Consulting,15096.33,INR,Bank Transfer,CC500,Completed,INV-21395
TXN0002,2025-08-05,Accounts Payable,Invoice Payment,ABC Technologies,8799.72,INR,Credit Card,CC200,Completed,INV-95181
TXN0003,2025-10-07,Accounts Receivable,Bank Transfer,ABC Technologies,205.55,USD,UPI,CC400,Completed,INV-54597
TXN0004,not_a_date,Accounts Receivable,Customer Receipt,Apex Services,36119.04,INR,Bank Transfer,CC300,Completed,INV-55082
TXN0005,2025-05-16,Procurement,Invoice Payment,Apex Services,97.14,USD,Bank Transfer,CC500,Completed,INV-48427
TXN0006,2025-07-05,Accounts Receivable,Expense Reimbursement,Metro Utilities,8111.06,INR,NEFT,CC100,Completed,INV-40512
TXN0007,2025-02-21,Payroll,Bank Transfer,Summit Office Products,453.72,USD,UPI,CC300,Completed,INV-56566
TXN0008,2025-04-18,Payroll,Journal Entry,BluePeak Consulting,19002.38,INR,Credit Card,CC200,Completed,INV-31417
TXN0009,2025-08-25,Payroll,Bank Transfer,Summit Office Products,73.38,EUR,Bank Transfer,CC200,Completed,INV-14207
TXN0010,2025-06-11,Payroll,Bank Transfer,Summit Office Products,15696.23,INR,UPI,CC400,Completed,INV-61856


In [0]:
#Import required functions
from pyspark.sql.functions import (
    col, trim, upper, initcap,
    to_date, when, count, sum as spark_sum
)

In [0]:
#Standardize text columns
df_clean = (
    df
    .withColumn("department", initcap(trim(col("department"))))
    .withColumn("transaction_type", initcap(trim(col("transaction_type"))))
    .withColumn("vendor_customer", trim(col("vendor_customer")))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn("currency", upper(trim(col("currency"))))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("cost_center", upper(trim(col("cost_center"))))
)

In [0]:
#Convert dates and amounts to correct data types
df_clean = (
    df_clean
    .withColumn(
        "transaction_date",
        to_date(col("transaction_date"), "yyyy-MM-dd")
    )
    .withColumn(
        "amount",
        col("amount").cast("double")
    ))

In [0]:
#Handle missing values
df_clean = df_clean.withColumn(
    "data_quality_status",
    when(
        col("transaction_id").isNull() |
        col("transaction_date").isNull() |
        col("amount").isNull() |
        col("vendor_customer").isNull() |
        col("cost_center").isNull(),
        "Needs Review"
    ).otherwise("Valid")
)

In [0]:
#Remove duplicate records
df_clean = df_clean.dropDuplicates()

In [0]:
#Validate financial amounts
df_clean = df_clean.withColumn(
    "amount_validation",
    when(col("amount").isNull(), "Missing Amount")
    .when(col("amount") < 0, "Review Negative Amount")
    .otherwise("Valid")
)

In [0]:
print(df.columns)
df.show(5, truncate=False)

['transaction_id', 'transaction_date', 'department', 'transaction_type', 'vendor_customer', 'amount', 'currency', 'payment_method', 'cost_center', 'status', 'invoice_number']
+--------------+----------------+-------------------+----------------+-------------------+--------+--------+--------------+-----------+---------+--------------+
|transaction_id|transaction_date|department         |transaction_type|vendor_customer    |amount  |currency|payment_method|cost_center|status   |invoice_number|
+--------------+----------------+-------------------+----------------+-------------------+--------+--------+--------------+-----------+---------+--------------+
|TXN0001       |2025-02-27      |Procurement        |Invoice Payment |BluePeak Consulting|15096.33|INR     |Bank Transfer |CC500      |Completed|INV-21395     |
|TXN0002       |2025-08-05      |Accounts Payable   |Invoice Payment |ABC Technologies   |8799.72 |INR     |Credit Card   |CC200      |Completed|INV-95181     |
|TXN0003       |2025

In [0]:
from pyspark.sql.functions import col, to_date, when

df_clean = df_clean.withColumn(
    "transaction_date",
    when(
        col("transaction_date").cast("string").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        to_date(col("transaction_date").cast("string"), "yyyy-MM-dd")
    ).otherwise(None)
)

In [0]:
#CHANGE not a date
from pyspark.sql.functions import col, expr

df_clean = df.withColumn(
    "transaction_date",
    expr("try_to_date(CAST(transaction_date AS STRING), 'yyyy-MM-dd')")
)

In [0]:
df.show(10)

+--------------+----------------+-------------------+--------------------+--------------------+--------+--------+--------------+-----------+---------+--------------+
|transaction_id|transaction_date|         department|    transaction_type|     vendor_customer|  amount|currency|payment_method|cost_center|   status|invoice_number|
+--------------+----------------+-------------------+--------------------+--------------------+--------+--------+--------------+-----------+---------+--------------+
|       TXN0001|      2025-02-27|        Procurement|     Invoice Payment| BluePeak Consulting|15096.33|     INR| Bank Transfer|      CC500|Completed|     INV-21395|
|       TXN0002|      2025-08-05|   Accounts Payable|     Invoice Payment|    ABC Technologies| 8799.72|     INR|   Credit Card|      CC200|Completed|     INV-95181|
|       TXN0003|      2025-10-07|Accounts Receivable|       Bank Transfer|    ABC Technologies|  205.55|     USD|           UPI|      CC400|Completed|     INV-54597|
|   

In [0]:
df_clean.select("transaction_id", "transaction_date").show(10, truncate=False)

+--------------+----------------+
|transaction_id|transaction_date|
+--------------+----------------+
|TXN0001       |2025-02-27      |
|TXN0002       |2025-08-05      |
|TXN0003       |2025-10-07      |
|TXN0004       |NULL            |
|TXN0005       |2025-05-16      |
|TXN0006       |2025-07-05      |
|TXN0007       |2025-02-21      |
|TXN0008       |2025-04-18      |
|TXN0009       |2025-08-25      |
|TXN0010       |2025-06-11      |
+--------------+----------------+
only showing top 10 rows


In [0]:
#Save the Silver table again
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("finance_transactions_silver")

In [0]:
#Verify the saved table
silver_df = spark.table("finance_transactions_silver")

print("Total rows:", silver_df.count())
display(silver_df.limit(10))

Total rows: 500


transaction_id,transaction_date,department,transaction_type,vendor_customer,amount,currency,payment_method,cost_center,status,invoice_number
TXN0001,2025-02-27,Procurement,Invoice Payment,BluePeak Consulting,15096.33,INR,Bank Transfer,CC500,Completed,INV-21395
TXN0002,2025-08-05,Accounts Payable,Invoice Payment,ABC Technologies,8799.72,INR,Credit Card,CC200,Completed,INV-95181
TXN0003,2025-10-07,Accounts Receivable,Bank Transfer,ABC Technologies,205.55,USD,UPI,CC400,Completed,INV-54597
TXN0004,null,Accounts Receivable,Customer Receipt,Apex Services,36119.04,INR,Bank Transfer,CC300,Completed,INV-55082
TXN0005,2025-05-16,Procurement,Invoice Payment,Apex Services,97.14,USD,Bank Transfer,CC500,Completed,INV-48427
TXN0006,2025-07-05,Accounts Receivable,Expense Reimbursement,Metro Utilities,8111.06,INR,NEFT,CC100,Completed,INV-40512
TXN0007,2025-02-21,Payroll,Bank Transfer,Summit Office Products,453.72,USD,UPI,CC300,Completed,INV-56566
TXN0008,2025-04-18,Payroll,Journal Entry,BluePeak Consulting,19002.38,INR,Credit Card,CC200,Completed,INV-31417
TXN0009,2025-08-25,Payroll,Bank Transfer,Summit Office Products,73.38,EUR,Bank Transfer,CC200,Completed,INV-14207
TXN0010,2025-06-11,Payroll,Bank Transfer,Summit Office Products,15696.23,INR,UPI,CC400,Completed,INV-61856


In [0]:
#Create the Gold summary
from pyspark.sql.functions import col, count, sum as spark_sum, round

# Read the cleaned Silver table
silver_df = spark.table("finance_transactions_silver")

# Create a business summary
gold_df = (
    silver_df
    .filter(
        col("department").isNotNull() &
        col("currency").isNotNull() &
        col("amount").isNotNull()
    )
    .groupBy("department", "currency")
    .agg(
        count("transaction_id").alias("total_transactions"),
        round(spark_sum("amount"), 2).alias("total_amount")
    )
)

display(gold_df)

department,currency,total_transactions,total_amount
Procurement,INR,46,504860.12
Accounts Payable,INR,51,584310.65
Accounts Receivable,USD,12,2011.15
Accounts Receivable,INR,43,933540.28
Procurement,USD,17,3047.59
Payroll,USD,11,1891.51
Payroll,INR,48,892441.33
Payroll,EUR,16,1864.58
accounts payable,EUR,1,134.33
Treasury,INR,58,799478.63


In [0]:
#Saved gold table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("finance_transactions_gold")

In [0]:
#Varify the table
gold_check = spark.table("finance_transactions_gold")

print("Gold table rows:", gold_check.count())
display(gold_check.orderBy("department", "currency"))

Gold table rows: 23


department,currency,total_transactions,total_amount
accounts payable,EUR,1,134.33
accounts payable,USD,1,683.2
Accounts Payable,EUR,15,2179.93
Accounts Payable,INR,51,584310.65
Accounts Payable,USD,20,2337.52
Accounts Receivable,EUR,21,2842.63
Accounts Receivable,INR,43,933540.28
Accounts Receivable,USD,12,2011.15
General Ledger,EUR,16,2725.95
General Ledger,INR,41,575858.1
